# Notebook 03 — FDIC Data Quality & Clean Bank–Quarter Panel

**Final Individual Report · Part 2 (20 points) · Consolidated 03A + 03B**

This notebook integrates the **47-field Raw Data Quality Audit (03A)** and **evidence-led cleaning / clean-panel construction (03B)**. It reads the immutable Part 1 Financials and auxiliary Parquet data and produces versioned Part 2 outputs. The FDIC **History extension (03C) is maintained separately**: its verified exit classifications are linked in the final section below.

**Important if you have already run 03A, 03B and 03C:** Combining notebooks is a documentation/code-maintenance step. **You do not need to run all cells again** merely to save the consolidated `.ipynb`. Your existing reports and Parquet files remain on Google Drive. Use the following optional **Existing-results checkpoint** to inspect them without re-running QA or cleaning.

**Execution from scratch:** Run all cells top to bottom if you specifically want a fully reproduced execution. This can take time: the 03A audit re-reads 43 yearly Parquet files, and 03B rewrites the versioned clean-panel outputs; it does **not** re-download the 170 quarters from the FDIC API. The 03C History download is not included here.

**Source safety:** No cell updates the original `data_raw/fdic/`, `data_storage/financials/year_partitioned_full/`, or the Part 1 manifest. Part 2 outputs are written into `outputs/part2/` and `data_processed/`. A full rerun **will replace** some Part 2 versioned outputs.

## 0. Optional quick check of **existing** Part 2 results (no full rerun)

Run **only the next cell** to inspect the previously generated 03A, 03B and 03C outputs. This standalone check mounts Google Drive and reads summaries/file metadata; it does **not** run transformations or rewrite any files. It is not a substitute for a full end-to-end reproducibility run.

In [ ]:
# SAFE STANDALONE CHECK: no API calls, no file writes, no full Parquet scans.
import os, json
from pathlib import Path

_root_override = os.environ.get('FDIC_PROJECT_ROOT')
if _root_override:
    _check_root = Path(_root_override).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    _check_root = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')

_paths = {
    '03A raw QA summary': _check_root / 'outputs/part2/raw_qa/audit_summary.json',
    '03B cleaning summary': _check_root / 'outputs/part2/cleaning_v1/cleaning_summary.json',
    '03B clean panel': _check_root / 'data_processed/clean_panel_v1',
    '03C History summary': _check_root / 'outputs/part2/history_exit_v1/history_exit_summary.json',
    '03C revised outcomes CSV': _check_root / 'outputs/part2/history_exit_v1/bank_exit_outcomes_with_history.csv',
    '03C revised outcomes Parquet': _check_root / 'data_processed/part2_history_exit_reference_v1/bank_exit_outcomes_with_history.parquet',
}
print('Project:', _check_root)
for _label, _path in _paths.items():
    print(f'{_label}:', 'FOUND' if _path.exists() else 'MISSING', '|', _path.relative_to(_check_root))

_qa = _paths['03A raw QA summary']
_clean = _paths['03B cleaning summary']
_hist = _paths['03C History summary']
if _qa.is_file():
    _d = json.loads(_qa.read_text(encoding='utf-8'))
    print('03A | financial rows:', _d.get('financial_rows'), '| fields:', _d.get('financial_fields_observed'))
if _clean.is_file():
    _d = json.loads(_clean.read_text(encoding='utf-8'))
    print('03B | rows before/after:', _d.get('rows_before'), '/', _d.get('rows_after'),
          '| acceptance:', _d.get('acceptance_passed'))
if _paths['03B clean panel'].is_dir():
    _files = list(_paths['03B clean panel'].glob('YEAR=*/data.parquet'))
    print('03B | clean-panel year files:', len(_files), '/ 43')
if _hist.is_file():
    _d = json.loads(_hist.read_text(encoding='utf-8'))
    print('03C | History events:', _d.get('history_events_collected'),
          '| verified years:', _d.get('verified_year_slices'),
          '| verified mergers:', _d.get('new_history_verified_nonassisted_mergers'),
          '| acceptance:', _d.get('acceptance_all_passed'))
print('Checkpoint complete. This cell does not modify datasets or reports.')

---
# Part A — Comprehensive Raw Data Quality Audit (03A)

Retain the original diagnostic implementation and reports under `outputs/part2/raw_qa/`. This section only profiles, audits and reports; it does not clean or overwrite Part 1 data.

## A0. Install dependencies and connect Drive

If running locally for a smoke test, set `FDIC_PROJECT_ROOT` to a test-project directory. On Colab, the notebook mounts your normal Google Drive folder.

In [ ]:
%pip -q install "polars>=1.20,<2" "pandas>=2.2,<3" "pyarrow>=17,<24" "psutil>=5.9"

In [ ]:
import os, json, math, time, sys, platform, zipfile, hashlib, re, shutil, tempfile, bisect
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import polars as pl

root_override = os.environ.get('FDIC_PROJECT_ROOT')
if root_override:
    PROJECT_ROOT = Path(root_override).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')

assert PROJECT_ROOT.exists(), f'Project folder missing: {PROJECT_ROOT}'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PyArrow:', pa.__version__, '| pandas:', pd.__version__)

## A1. Paths, study scope, and read-only guarantees

Part 1 writes Financials to `data_storage/financials/year_partitioned_full/YEAR=YYYY/data.parquet` and auxiliary snapshots under `data_raw/fdic/aux_pages/`. Notebook 03 reads these exactly as produced by Notebook 02.

In [ ]:
START_QUARTER = '19840331'
END_QUARTER = '20260630'
FIN_DIR = PROJECT_ROOT / 'data_storage/financials/year_partitioned_full'
RAW_DIR = PROJECT_ROOT / 'data_raw/fdic'
AUX_DIR = RAW_DIR / 'aux_pages'
MANIFEST_PATH = RAW_DIR / 'manifest_colab_v1.json'
PART1_DIR = PROJECT_ROOT / 'outputs/part1'
OUT_DIR = PROJECT_ROOT / 'outputs/part2/raw_qa'
OUT_DIR.mkdir(parents=True, exist_ok=True)  # ONLY report outputs are writable

assert FIN_DIR.is_dir(), f'FULL Financials year partitions not found: {FIN_DIR}'
assert MANIFEST_PATH.is_file(), f'Part 1 manifest not found: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest.get('version') == 1, 'Unexpected Part 1 manifest version'
entries = manifest.get('entries', {})

def quarter_end_strings(first, last):
    periods = pd.period_range(pd.Period(first[:4] + 'Q' + str((int(first[4:6])-1)//3+1)),
                              pd.Period(last[:4] + 'Q' + str((int(last[4:6])-1)//3+1)),
                              freq='Q')
    return [p.end_time.strftime('%Y%m%d') for p in periods]

EXPECTED_QUARTERS = quarter_end_strings(START_QUARTER, END_QUARTER)
Q_ORD = {q:i for i,q in enumerate(EXPECTED_QUARTERS)}
YEAR_FILES = sorted(FIN_DIR.glob('YEAR=*/data.parquet'), key=lambda p:int(p.parent.name.split('=')[1]))
print('Year partitions located:', len(YEAR_FILES))
print('Expected quarters:', len(EXPECTED_QUARTERS), EXPECTED_QUARTERS[0], '→', EXPECTED_QUARTERS[-1])
print('Auxiliary source summary:', {s:entries.get(f'aux:{s}:summary',{}).get('status','MISSING')
                                    for s in ('failures','institutions')})
assert YEAR_FILES, 'No curated FULL year Parquet files found; check FIN_DIR.'

# No cell in this notebook invokes FDIC API or writes to FIN_DIR, RAW_DIR, MANIFEST_PATH.

## A2. Inventory and source-side expectations

Build an independent inventory from Parquet metadata. Compare the observed quarterly counts to the acquisition manifest. Discrepancies are **flags**, not automatically corrected.

In [ ]:
manifest_expected = {q:entries.get('financials:'+q, {}) for q in EXPECTED_QUARTERS}
known_financial_records = sum(int(v.get('rows') or 0) for v in manifest_expected.values())
if len(YEAR_FILES) != 43:
    print(f'WARNING: expected 43 year partitions for FULL, observed {len(YEAR_FILES)}')
if any(v.get('status') not in {'VERIFIED','VERIFIED_SCHEMA_GAP'} for v in manifest_expected.values()):
    print('WARNING: not all 170 quarters have VERIFIED status in the manifest')

inventory=[]
for f in YEAR_FILES:
    pf=pq.ParquetFile(f)
    inventory.append({'year':int(f.parent.name.split('=')[1]),
        'path':str(f.relative_to(PROJECT_ROOT)),
        'rows_in_parquet_metadata':int(pf.metadata.num_rows),
        'column_count':len(pf.schema_arrow),
        'size_mib':round(f.stat().st_size/1024**2,3),
        'schema_text':str(pf.schema_arrow)[:250]})
print('Parquet metadata row total:', sum(x['rows_in_parquet_metadata'] for x in inventory))
print('Part 1 manifest row total:', known_financial_records)
assert len({x['year'] for x in inventory}) == len(inventory), 'Duplicate YEAR directories!'

## A3. Financials — comprehensive raw data profiling (all available fields)

This diagnostic scans **every Financials field** in the observed Parquet schema, year by year (including IDs, dates and categorical fields). It checks missingness and schema drift for all fields; numerical flags, percentiles and extremes for all numerical financial fields; blank strings and distinct values for non-numerical fields; and primary key integrity / reporting gaps.

**Important:** CERT is a numeric *identifier*, not a continuous financial metric. Business-specific plausibility tests remain selective because zeros and negatives have different meanings across accounting fields. This cell does not clean or overwrite source data.


In [ ]:
# Comprehensive Financials QA: all columns are profiled; no raw/curated data are changed.
# Requires notebook setup cells (pa, pc, pq, pd, np, Counter, YEAR_FILES, Q_ORD).

NONNEGATIVE_REVIEW_FIELDS = [
    'ASSET', 'DEP', 'DEPDOM', 'COREDEP', 'DEPINS', 'DEPUNINS', 'CHBAL', 'SC',
    'LNLSGR', 'LNLSNET', 'NCLNLS', 'P3ASSET', 'P9ASSET', 'NAASSET', 'NTLNLS'
]
ZERO_DENOMINATOR_FIELDS = ['ASSET', 'LNLSGR', 'DEP']
CANDIDATE_RATIO_FIELDS = ['ROA', 'ROE', 'NIMY', 'RBC1AAJ', 'RBC1RWAJ', 'RBCRWAJ', 'CBLRIND']
PAIRWISE_REVIEW = [('DEP', 'ASSET'), ('LNLSGR', 'ASSET'), ('EQ', 'ASSET')]
EXPECTED_NUMERIC_FIELDS = set(
    NONNEGATIVE_REVIEW_FIELDS + ZERO_DENOMINATOR_FIELDS +
    CANDIDATE_RATIO_FIELDS + ['EQ', 'NETINC']
)
# Identifiers are audited as keys, not treated as continuous financial measurements.
NUMERIC_IDENTIFIER_FIELDS = {'CERT'}

field_year = []
year_key = []
year_dates = []
year_numeric = []
year_non_numeric = []
year_pair = []
year_distributions = []
quarter_counts = Counter()
schema_by_year = []
key_examples = []
bank_history = {}
first_schema = None
TOTAL_ROWS = 0
all_fields_seen = set()
all_financial_numeric_fields = set()
all_non_numeric_fields = set()


def count_true(condition):
    if condition is None:
        return 0
    result = pc.sum(pc.cast(pc.fill_null(condition, False), pa.int64())).as_py()
    return int(result or 0)


def normalized_q(value):
    if value is None:
        return None
    if isinstance(value, bytes):
        value = value.decode('utf-8', 'replace')
    return str(value).strip()


def is_numeric_arrow(dtype):
    return (pa.types.is_integer(dtype) or pa.types.is_floating(dtype)
            or pa.types.is_decimal(dtype))


for year_i, f in enumerate(YEAR_FILES, start=1):
    year = int(f.parent.name.split('=')[1])
    # Read only one year at a time; no source file is overwritten.
    tab = pq.ParquetFile(f).read()
    n = tab.num_rows
    TOTAL_ROWS += n
    schema = tab.schema
    if first_schema is None:
        first_schema = schema
    names = schema.names
    all_fields_seen.update(names)

    schema_by_year.append({
        'year': year, 'field_count': len(names),
        'missing_vs_first_year': ';'.join(sorted(set(first_schema.names) - set(names))),
        'additional_vs_first_year': ';'.join(sorted(set(names) - set(first_schema.names))),
        'type_drift': ';'.join(sorted(
            name for name in set(first_schema.names) & set(names)
            if first_schema.field(name).type != schema.field(name).type
        ))
    })
    if not {'CERT', 'REPDTE'} <= set(names):
        raise ValueError(f'Critical key fields absent: {f}')

    # Tier 1: completeness and data type for EVERY field, including keys and strings.
    for fld in schema:
        col = tab.column(fld.name)
        field_year.append({
            'year': year, 'field': fld.name, 'dtype': str(fld.type),
            'rows': n, 'null_count': int(col.null_count),
            'null_pct': round(100 * col.null_count / n, 5) if n else None
        })
        if not is_numeric_arrow(fld.type):
            all_non_numeric_fields.add(fld.name)
            values = col.to_pylist()
            non_null_values = [v for v in values if v is not None]
            blank_count = sum(
                isinstance(v, (str, bytes)) and not normalized_q(v)
                for v in non_null_values
            )
            year_non_numeric.append({
                'year': year, 'field': fld.name, 'dtype': str(fld.type),
                'rows': n, 'non_null': len(non_null_values),
                'null_count': int(col.null_count),
                'blank_or_whitespace_count': int(blank_count),
                'distinct_non_null_values': len({str(v) for v in non_null_values}),
                'expected_numeric_but_wrong_type': fld.name in EXPECTED_NUMERIC_FIELDS
            })

    # Primary keys, quarter alignment and reporting continuity.
    certs = tab.column('CERT').to_pylist()
    dates = [normalized_q(x) for x in tab.column('REPDTE').to_pylist()]
    invalid_cert = sum(x is None for x in certs)
    invalid_date = sum(q not in Q_ORD for q in dates)
    wrong_year = sum(q in Q_ORD and int(q[:4]) != year for q in dates)
    local_pairs = Counter(zip(certs, dates))
    local_duplicates = sum(v - 1 for v in local_pairs.values() if v > 1)
    if local_duplicates and len(key_examples) < 100:
        for (c, q), freq in local_pairs.items():
            if freq > 1 and len(key_examples) < 100:
                key_examples.append({'year': year, 'CERT': c, 'REPDTE': q, 'count': freq})
    year_key.append({
        'year': year, 'rows': n, 'null_CERT': invalid_cert,
        'invalid_REPDTE': invalid_date, 'wrong_partition_year': wrong_year,
        'duplicate_CERT_REPDTE_rows': local_duplicates
    })
    valid_qs = Counter(q for q in dates if q in Q_ORD)
    quarter_counts.update(valid_qs)
    year_dates.append({
        'year': year, 'distinct_valid_quarters': len(valid_qs),
        'quarter_values': ';'.join(sorted(valid_qs)),
        'outside_expected_quarter_dates': invalid_date,
        'wrong_year_rows': wrong_year
    })

    for c, q in sorted(zip(certs, dates), key=lambda x: (
        Q_ORD.get(x[1], 999999), -1 if x[0] is None else x[0]
    )):
        if c is None or q not in Q_ORD:
            continue
        ordq = Q_ORD[q]
        state = bank_history.get(c)
        if state is None:
            bank_history[c] = {
                'CERT': c, 'first_q': q, 'last_q': q,
                'first_ord': ordq, 'last_ord': ordq,
                'observed_quarters': 1, 'gap_events': 0, 'gap_quarters': 0
            }
        elif ordq > state['last_ord']:
            delta = ordq - state['last_ord']
            if delta > 1:
                state['gap_events'] += 1
                state['gap_quarters'] += delta - 1
            state['last_ord'] = ordq
            state['last_q'] = q
            state['observed_quarters'] += 1

    # Tier 1: numerical diagnostics for EVERY available numeric financial field.
    # CERT is already audited as an identifier above.
    numeric_fields = sorted(
        fld.name for fld in schema
        if is_numeric_arrow(fld.type) and fld.name not in NUMERIC_IDENTIFIER_FIELDS
    )
    all_financial_numeric_fields.update(numeric_fields)

    for name in numeric_fields:
        col = tab.column(name)
        nonfinite = count_true(pc.invert(pc.is_finite(col))) if pa.types.is_floating(col.type) else 0
        zero_count = count_true(pc.equal(col, 0))
        negative_count = count_true(pc.less(col, 0))
        year_numeric.append({
            'year': year, 'field': name, 'error': '',
            'non_null': int(n - col.null_count),
            'zero_count': zero_count,
            'negative_count': negative_count,
            'nonfinite_count': nonfinite,
            'negative_requires_review_by_semantics': name in NONNEGATIVE_REVIEW_FIELDS,
            'zero_denominator_candidate': name in ZERO_DENOMINATOR_FIELDS
        })

        # Percentiles per year. Ignore null, NaN and +/-Inf; do not clip or impute.
        values = pd.to_numeric(pd.Series(col.to_pylist()), errors='coerce')
        values = values.replace([np.inf, -np.inf], np.nan).dropna()
        if len(values):
            quantiles = values.quantile([0.01, 0.50, 0.99])
            year_distributions.append({
                'year': year, 'field': name, 'non_null_finite': int(len(values)),
                'min': float(values.min()), 'p01': float(quantiles.loc[0.01]),
                'median': float(quantiles.loc[0.50]),
                'p99': float(quantiles.loc[0.99]), 'max': float(values.max())
            })
        del values

    # Retain type-error diagnostics for business fields that ought to be numeric.
    for name in sorted(EXPECTED_NUMERIC_FIELDS & set(names)):
        if name not in numeric_fields:
            col = tab.column(name)
            year_numeric.append({
                'year': year, 'field': name, 'error': 'not_numeric',
                'non_null': int(n - col.null_count),
                'zero_count': None, 'negative_count': None,
                'nonfinite_count': None,
                'negative_requires_review_by_semantics': name in NONNEGATIVE_REVIEW_FIELDS,
                'zero_denominator_candidate': name in ZERO_DENOMINATOR_FIELDS
            })

    # Tier 2: business checks only where numerator/denominator comparisons make sense.
    for numer, denom in PAIRWISE_REVIEW:
        if numer in names and denom in names and all(
            is_numeric_arrow(tab.column(k).type) for k in (numer, denom)
        ):
            condition = pc.and_(
                pc.greater(tab.column(numer), tab.column(denom)),
                pc.greater(tab.column(denom), 0)
            )
            year_pair.append({
                'year': year, 'numerator': numer, 'denominator': denom,
                'numerator_gt_denominator_count': count_true(condition)
            })

    del tab
    if year_i == 1 or year_i % 5 == 0 or year_i == len(YEAR_FILES):
        print(
            f'[Comprehensive Financial QA] {year_i}/{len(YEAR_FILES)} years'
            f' | {TOTAL_ROWS:,} rows', flush=True
        )

print('FINANCIALS AUDIT COMPLETE | rows:', f'{TOTAL_ROWS:,}',
      '| unique banks:', len(bank_history))
print('All detected Financials fields:', len(all_fields_seen))
print('Numeric financial fields audited:', len(all_financial_numeric_fields))
print('Non-numeric fields audited:', len(all_non_numeric_fields))
print('Numeric identifier CERT audited separately as a key.')


## A4. Independent 170-quarter reconciliation, missingness and bank continuity

**Interpretation:** a gap in a bank's quarterly reports is an anomaly for investigation, **not proof of merger, failure, or disappearance**. Year-level missingness reveals historical schema/availability issues that all-years averages conceal.

In [ ]:
quarter_audit=[]
for q in EXPECTED_QUARTERS:
    exp=manifest_expected[q]
    expected_rows=exp.get('rows')
    observed=int(quarter_counts.get(q,0))
    expected_int=int(expected_rows) if expected_rows is not None else None
    quarter_audit.append({'quarter':q,'year':int(q[:4]),'observed_rows':observed,
        'manifest_rows':expected_int,'delta_vs_manifest':observed-expected_int if expected_int is not None else None,
        'manifest_status':exp.get('status','NOT_ATTEMPTED'),
        'record_match':(observed==expected_int) if expected_int is not None else False})

missing_years=sorted(set(int(q[:4]) for q in EXPECTED_QUARTERS)-{x['year'] for x in inventory})
all_null_year=pd.DataFrame(field_year)
by_field=(all_null_year.groupby(['field','dtype'],dropna=False,as_index=False)
          .agg(rows=('rows','sum'),null_count=('null_count','sum')))
by_field['null_pct']=(100*by_field['null_count']/by_field['rows']).round(5)
by_field=by_field.sort_values(['null_pct','field'],ascending=[False,True]).reset_index(drop=True)

bank_gaps=[{'CERT':v['CERT'],'first_quarter':v['first_q'],
            'last_quarter':v['last_q'],'observed_quarters':v['observed_quarters'],
            'gap_events':v['gap_events'],'gap_quarters':v['gap_quarters']}
           for v in bank_history.values() if v['gap_events']>0]

print('Expected vs observed quarter mismatches:',sum(not r['record_match'] for r in quarter_audit))
print('Missing years:',missing_years)
print('Duplicated bank-quarter rows:',sum(x['duplicate_CERT_REPDTE_rows'] for x in year_key))
print('Banks with reporting gaps:',len(bank_gaps))
print('Top 12 fields by missing rate:')
display(by_field.head(12))

## A5. Load only the verified auxiliary snapshots (Failures & Institutions)

A page from an **old snapshot** may exist on Drive. Select pages through the manifest's currently verified `aux:{source}:summary.snapshot` value, not a broad glob over all snapshots. Check `API_RECORD_ID`, missing `CERT`, and date parsing.

In [ ]:
aux_tables={}; aux_diagnostics=[]
for source in ['failures','institutions']:
    summary=entries.get(f'aux:{source}:summary',{})
    if summary.get('status') not in {'VERIFIED','VERIFIED_SCHEMA_GAP'}:
        raise RuntimeError(f'{source}: current auxiliary snapshot is not verified: {summary}')
    snapshot=summary.get('snapshot')
    folder=AUX_DIR/source/f'snapshot={snapshot}'
    paths=sorted(folder.glob('page_*.parquet'))
    assert paths, f'No active {source} pages at {folder}'
    if summary.get('pages') is not None:
        assert len(paths)==int(summary['pages']), f'{source}: expected {summary["pages"]} pages, got {len(paths)}'
    # Only ~4k + ~28k records: safe to hold auxiliary tables in memory.
    pieces=[pq.ParquetFile(p).read() for p in paths]
    table=pa.concat_tables(pieces, promote_options='none')
    aux_tables[source]=table
    ids=table.column('API_RECORD_ID').to_pylist() if 'API_RECORD_ID' in table.column_names else []
    valid=[x for x in ids if x is not None and str(x).strip()]
    certs=table.column('CERT').to_pylist()
    diag={'source':source,'pages':len(paths),'rows':table.num_rows,
       'expected_api_total':summary.get('api_total'),
       'missing_CERT':sum(c is None for c in certs),
       'missing_API_RECORD_ID':table.num_rows-len(valid),
       'duplicate_API_RECORD_ID':len(valid)-len(set(valid)),
       'unique_CERT_excluding_null':len(set(c for c in certs if c is not None))}
    aux_diagnostics.append(diag)
    print(source,diag)
    del pieces

fdf=aux_tables['failures'].select([k for k in ('CERT','FAILDATE','FAILYR','NAME','API_RECORD_ID')
                                 if k in aux_tables['failures'].column_names]).to_pandas()
idf=aux_tables['institutions'].select([k for k in ('CERT','ACTIVE','ENDEFYMD','ESTYMD','CHANGEC1','API_RECORD_ID')
                                     if k in aux_tables['institutions'].column_names]).to_pandas()

def parse_fdic_dates(series):
    return pd.to_datetime(series.astype('string').str.strip(),format='mixed',errors='coerce')

fdf['FAILDATE_PARSED']=parse_fdic_dates(fdf['FAILDATE'])
idf['ENDEFYMD_PARSED']=parse_fdic_dates(idf['ENDEFYMD']) if 'ENDEFYMD' in idf else pd.NaT
fdf['CERT_ID']=pd.to_numeric(fdf['CERT'],errors='coerce').astype('Int64')
idf['CERT_ID']=pd.to_numeric(idf['CERT'],errors='coerce').astype('Int64')
print('Failures invalid/missing dates:',int(fdf['FAILDATE_PARSED'].isna().sum()))
print('Institutions invalid/missing ENDEFYMD:',int(idf['ENDEFYMD_PARSED'].isna().sum()))

## A6. Failure linking and exit diagnostics (retrospective only)

Check how many FDIC failures (including pre-1984 records) can link to study banks using `CERT`. This is a **linkability report**, not a training label. A bank's last observed quarter does not itself identify the exit cause. Institution `ACTIVE` and end-date fields are snapshot information; using them in features for historical quarters would leak information.

In [ ]:
PANEL_START=pd.Timestamp('1984-01-01')
PANEL_END=pd.Timestamp('2026-06-30')
fdf['within_study_window']=fdf['FAILDATE_PARSED'].between(PANEL_START,PANEL_END,inclusive='both')
fdf['cert_in_financials']=fdf['CERT_ID'].isin(set(bank_history))
fdf['event_quarter']=fdf['FAILDATE_PARSED'].apply(
    lambda d: d.to_period('Q').end_time.strftime('%Y%m%d') if pd.notna(d) else None)
fdf['bank_first_quarter']=fdf['CERT_ID'].map({c:b['first_q'] for c,b in bank_history.items()})
fdf['bank_last_quarter']=fdf['CERT_ID'].map({c:b['last_q'] for c,b in bank_history.items()})
fdf['event_q_offset_from_last']=fdf.apply(
    lambda r: ((r['FAILDATE_PARSED'].year-1984)*4+(r['FAILDATE_PARSED'].quarter-1)
              - Q_ORD[r['bank_last_quarter']])
    if pd.notna(r['FAILDATE_PARSED']) and r['bank_last_quarter'] in Q_ORD else np.nan,
    axis=1)

fail_by_cert=defaultdict(list)
for row in fdf.itertuples(index=False):
    if pd.notna(row.CERT_ID) and pd.notna(row.FAILDATE_PARSED):
        fail_by_cert[int(row.CERT_ID)].append(row.FAILDATE_PARSED)

inst_by_cert={}
for row in idf.itertuples(index=False):
    if pd.notna(row.CERT_ID):
        inst_by_cert[int(row.CERT_ID)]=row

exit_rows=[]
for cert,bank in sorted(bank_history.items()):
    failures=fail_by_cert.get(cert,[])
    in_scope=[d for d in failures if PANEL_START<=d<=PANEL_END]
    inst=inst_by_cert.get(cert)
    gap_to_failure=None
    if in_scope:
        offsets=[((d.year-1984)*4+(d.quarter-1))-bank['last_ord'] for d in in_scope]
        gap_to_failure=min(offsets,key=abs)
        if 0 <= gap_to_failure <= 4:
            exit_type='FAILURE_EVENT_NEAR_LAST_REPORT'
        else:
            exit_type='FAILURE_EVENT_TIMING_REVIEW'
    elif bank['last_ord']==len(EXPECTED_QUARTERS)-1:
        exit_type='RIGHT_CENSORED_AT_2026Q2'
    else:
        exit_type='NONFAILURE_EXIT_OR_REPORTING_GAP_UNVERIFIED'
    exit_rows.append({'CERT':cert,'first_quarter':bank['first_q'],
        'last_quarter':bank['last_q'],'observed_quarters':bank['observed_quarters'],
        'gap_quarters':bank['gap_quarters'],'failure_event_count_all_years':len(failures),
        'failure_event_count_study_window':len(in_scope),
        'failure_quarter_offset_from_last':gap_to_failure,
        'in_institutions_snapshot':inst is not None,
        'institutions_ACTIVE_CURRENT_SNAPSHOT_ONLY':getattr(inst,'ACTIVE',None) if inst else None,
        'exit_diagnostic':exit_type})

exit_counts=pd.DataFrame(exit_rows)['exit_diagnostic'].value_counts().to_dict()
fail_link_summary={
  'failure_records_all_years':int(len(fdf)),
  'failures_missing_CERT':int(fdf['CERT_ID'].isna().sum()),
  'failures_invalid_or_missing_FAILDATE':int(fdf['FAILDATE_PARSED'].isna().sum()),
  'failure_records_in_1984_2026Q2':int(fdf['within_study_window'].sum()),
  'failure_records_in_window_with_panel_CERT':int((fdf['within_study_window'] & fdf['cert_in_financials']).sum()),
  'failure_records_in_window_without_panel_CERT':int((fdf['within_study_window'] & ~fdf['cert_in_financials']).sum()),
  'records_with_failure_date_before_panel':int((fdf['FAILDATE_PARSED']<PANEL_START).sum()),
  'records_with_failure_date_after_panel':int((fdf['FAILDATE_PARSED']>PANEL_END).sum()),
}
print('Failure linking:',fail_link_summary)
print('Exit diagnostics:',exit_counts)
print('NOTE: NONFAILURE_EXIT_OR_REPORTING_GAP_UNVERIFIED is NOT a confirmed merger.')

## A7. Automatically generate issue register and decision questions

This table separates **measured evidence** from a proposed **handling policy**. Rules are not yet implemented. We decide cleaning after reading the diagnostics; the clean panel will be written into a new `data_processed/` location in a later notebook stage.

In [ ]:
qa_issues=[]
def issue(code,severity,count,decision,evidence):
    qa_issues.append({'issue_code':code,'severity':severity,'affected_count':int(count),
                      'handling_to_decide':decision,'evidence':evidence})

quarter_mismatch=sum(not r['record_match'] for r in quarter_audit)
issue('QUARTER_COUNT_MISMATCH','CRITICAL' if quarter_mismatch else 'PASS',quarter_mismatch,
      'Investigate source/partition mapping; do not auto-fill quarters',
      'audit_quarter_coverage.csv')
issue('MISSING_YEAR_PARTITION','CRITICAL' if missing_years else 'PASS',len(missing_years),
      'Recover only missing verified year partitions from Part 1', 'audit_inventory.csv')
dup_rows=sum(x['duplicate_CERT_REPDTE_rows'] for x in year_key)
issue('DUPLICATE_BANK_QUARTER','CRITICAL' if dup_rows else 'PASS',dup_rows,
      'Inspect exact duplicates vs conflicting records; choose deterministic rule','audit_key_integrity.csv')
null_keys=sum(x['null_CERT']+x['invalid_REPDTE']+x['wrong_partition_year'] for x in year_key)
issue('BAD_OR_MISPLACED_KEYS','CRITICAL' if null_keys else 'PASS',null_keys,
      'Identify malformed rows, quarantine if needed; do not guess identifiers','audit_key_integrity.csv')
drift_count=sum(bool(x['missing_vs_first_year'] or x['additional_vs_first_year'] or x['type_drift'])
               for x in schema_by_year)
issue('SCHEMA_DRIFT','REVIEW' if drift_count else 'PASS',drift_count,
      'Use compatible types; explain nonavailability instead of zero fill','audit_schema_by_year.csv')
for name in ['ASSET','DEP','EQ','LNLSGR','ROA','RBC1AAJ']:
    rr=by_field[by_field.field==name]
    if not rr.empty:
        nmiss=int(rr.iloc[0]['null_count'])
        issue('MISSING_'+name,'REVIEW' if nmiss else 'PASS',nmiss,
            'Document time pattern; training-only imputation later, if justified',
            'audit_missingness_by_year.csv')
num_df=pd.DataFrame(year_numeric)
# Comprehensive field diagnostics supplement the curated business issue register.
nonfinite_total=int(pd.to_numeric(num_df['nonfinite_count'],errors='coerce').fillna(0).sum())
issue('NONFINITE_NUMERIC_VALUES','REVIEW' if nonfinite_total else 'PASS',nonfinite_total,
      'Inspect affected values by field and year; do not automatically replace them',
      'audit_numeric_flags_by_year.csv')
nonnum_df=pd.DataFrame(year_non_numeric)
blank_total=int(pd.to_numeric(nonnum_df['blank_or_whitespace_count'],errors='coerce').fillna(0).sum()) if not nonnum_df.empty else 0
issue('BLANK_NONNUMERIC_VALUES','REVIEW' if blank_total else 'PASS',blank_total,
      'Review blank categorical/date values by field; do not treat all blanks as failures',
      'audit_non_numeric_flags_by_year.csv')
for name in ['ASSET','DEP','EQ','LNLSGR','NETINC']:
    group=num_df[num_df.field==name]
    if not group.empty:
        z=int(pd.to_numeric(group.zero_count,errors='coerce').fillna(0).sum())
        n=int(pd.to_numeric(group.negative_count,errors='coerce').fillna(0).sum())
        issue('ZERO_'+name,'REVIEW' if z else 'PASS',z,
            'Zero may invalidate a ratio denominator; preserve underlying observation',
            'audit_numeric_flags_by_year.csv')
        issue('NEGATIVE_'+name,'REVIEW' if n else 'PASS',n,
            'Assess accounting definition; negative EQ/NETINC may signal distress',
            'audit_numeric_flags_by_year.csv')
issue('BANK_REPORTING_GAPS','REVIEW' if bank_gaps else 'PASS',len(bank_gaps),
      'Investigate missing interim reports vs true institutional changes',
      'audit_bank_reporting_gaps.csv')
miss_ids=sum(x['missing_API_RECORD_ID'] for x in aux_diagnostics)
dup_ids=sum(x['duplicate_API_RECORD_ID'] for x in aux_diagnostics)
issue('AUX_RECORD_ID_PROBLEM','CRITICAL' if miss_ids+dup_ids else 'PASS',miss_ids+dup_ids,
      'Review current manifest snapshot; never infer missing ID from CERT','audit_auxiliary_quality.csv')
issue('UNMATCHED_STUDY_WINDOW_FAILURES','REVIEW' if fail_link_summary['failure_records_in_window_without_panel_CERT'] else 'PASS',
      fail_link_summary['failure_records_in_window_without_panel_CERT'],
      'Compare date coverage and CERT changes; do not force unmatched joins','audit_failure_event_links.csv')
issue('AMBIGUOUS_NONFAILURE_EXITS','REVIEW',exit_counts.get('NONFAILURE_EXIT_OR_REPORTING_GAP_UNVERIFIED',0),
      'Do not label all disappearances as mergers; History/event evidence needed','audit_bank_exit_candidates.csv')
issue('POINT_IN_TIME_AVAILABILITY','LIMITATION',0,
      'REPDTE is not release timestamp; use conservative lag and exclude modern snapshot states from features',
      'Notebook 02 schema / point-in-time discussion')

qa_issues.sort(key=lambda x:({'CRITICAL':0,'REVIEW':1,'LIMITATION':2,'PASS':3}.get(x['severity'],9),
                             -x['affected_count'],x['issue_code']))
print('Issues grouped by severity:',dict(Counter(i['severity'] for i in qa_issues)))
display(pd.DataFrame(qa_issues).head(20))

## A8. Export diagnostic reports and a ZIP for review

**Only files in `outputs/part2/raw_qa/` are written.** All source Parquet and manifest files stay unchanged. Send the ZIP back for a targeted set of cleaning rules and the next stage of building the clean bank–quarter panel.

In [ ]:
report_tables={
 'audit_inventory.csv':pd.DataFrame(inventory),
 'audit_quarter_coverage.csv':pd.DataFrame(quarter_audit),
 'audit_key_integrity.csv':pd.DataFrame(year_key),
 'audit_key_duplicate_examples.csv':pd.DataFrame(key_examples,columns=['year','CERT','REPDTE','count']),
 'audit_schema_by_year.csv':pd.DataFrame(schema_by_year),
 'audit_missingness_by_year.csv':pd.DataFrame(field_year),
 'audit_missingness_overall.csv':by_field,
 'audit_numeric_flags_by_year.csv':pd.DataFrame(year_numeric),
 'audit_non_numeric_flags_by_year.csv':pd.DataFrame(year_non_numeric),
 'audit_numeric_distributions_by_year.csv':pd.DataFrame(year_distributions),
 'audit_pairwise_flags_by_year.csv':pd.DataFrame(year_pair),
 'audit_bank_reporting_gaps.csv':pd.DataFrame(bank_gaps,
         columns=['CERT','first_quarter','last_quarter','observed_quarters','gap_events','gap_quarters']),
 'audit_auxiliary_quality.csv':pd.DataFrame(aux_diagnostics),
 'audit_failure_event_links.csv':fdf[['API_RECORD_ID','CERT','FAILDATE',
       'FAILDATE_PARSED','within_study_window','cert_in_financials',
       'event_quarter','bank_first_quarter','bank_last_quarter','event_q_offset_from_last']].copy(),
 'audit_bank_exit_candidates.csv':pd.DataFrame(exit_rows),
 'audit_issues.csv':pd.DataFrame(qa_issues),
}
for name,frame in report_tables.items():
    dest=OUT_DIR/name
    frame.to_csv(dest,index=False,encoding='utf-8-sig')
    print(f'Saved {name}: {len(frame):,} rows')

summary={
 'audit_type':'RAW_DATA_QUALITY_DIAGNOSIS_ONLY_NO_CLEANING',
 'generated_at_utc':datetime.now(timezone.utc).isoformat(),
 'project':str(PROJECT_ROOT),
 'input_curated_financials':str(FIN_DIR.relative_to(PROJECT_ROOT)),
 'input_manifest':str(MANIFEST_PATH.relative_to(PROJECT_ROOT)),
 'part1_manifest_last_updated_utc':manifest.get('last_updated_utc'),
 'financial_year_partitions':len(YEAR_FILES),
 'financial_rows':int(TOTAL_ROWS),
 'financial_fields_observed':len(all_fields_seen),
 'financial_numeric_fields_profiled':len(all_financial_numeric_fields),
 'financial_non_numeric_fields_profiled':len(all_non_numeric_fields),
 'numeric_identifier_fields':sorted(NUMERIC_IDENTIFIER_FIELDS & all_fields_seen),
 'financial_manifest_expected_rows':int(known_financial_records),
 'expected_quarters':len(EXPECTED_QUARTERS),
 'quarters_with_record_mismatch':int(quarter_mismatch),
 'missing_years':missing_years,
 'unique_financial_CERTs':len(bank_history),
 'duplicate_bank_quarter_rows':int(dup_rows),
 'bad_key_or_year_rows':int(null_keys),
 'banks_with_observed_quarter_gaps':len(bank_gaps),
 'schema_drift_years':int(drift_count),
 'failure_link_summary':fail_link_summary,
 'exit_diagnostic_counts':exit_counts,
 'auxiliary_diagnostics':aux_diagnostics,
 'issue_severity_counts':dict(Counter(i['severity'] for i in qa_issues)),
 'source_statuses':{s:entries.get(f'aux:{s}:summary',{}).get('status') for s in ('failures','institutions')},
 'methodology_cautions':[
   'No source data were modified',
   'Institution ACTIVE/ENDEFYMD refer to current snapshot, not point-in-time feature states',
   'REPDTE is reporting quarter end, not necessarily the public release timestamp',
   'Unmatched failure events and reporting gaps require event-identity review',
   'All flags are diagnostic, no winsorization or imputation performed'
 ],
 'software':{'python':sys.version.split()[0],'pyarrow':pa.__version__,
             'pandas':pd.__version__,'platform':platform.platform()}
}
summary_path=OUT_DIR/'audit_summary.json'
summary_path.write_text(json.dumps(summary,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
archive=OUT_DIR/'FDIC_Part2_Raw_QA_Reports.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob('audit_*')):
        if p.is_file() and p!=archive:
            zf.write(p,arcname=p.name)
print('\nAUDIT EXPORTED:',OUT_DIR)
print('UPLOAD THIS FILE FOR REVIEW:',archive)
print('ZIP size (MiB):',round(archive.stat().st_size/1024**2,2))

## A9. Interpretation and decision point

After running, check `audit_summary.json`, `audit_issues.csv`, and missingness by year, all-field numeric distributions and non-numeric flags. Upload the **ZIP archive** to ChatGPT, not raw 1.68-million-row files.

**Next (not implemented here):** agree on missing-data policies, key anomalies, bank exit classification (failure vs other exits vs censoring) and documented point-in-time lag; then write a separate cleaned bank–quarter panel to `data_processed/clean_panel/`. Part 3 should benchmark **the same accepted transformations** in Polars and pandas and demonstrate key parity steps in PySpark.

---
# Part B — Cleaning, event alignment, and clean bank–quarter panel (03B)

**Prerequisite:** Part A must have produced `outputs/part2/raw_qa/audit_summary.json` and `audit_bank_exit_candidates.csv`. When executing this merged notebook from top to bottom, these inputs are already available. The standalone 03B setup/install cells have been consolidated to avoid a second Drive mount, a second dependency installation, and conflicting `PROJECT_ROOT` definitions.

**Outputs:** `data_processed/clean_panel_v1/`, `data_processed/part2_event_reference_v1/` and `outputs/part2/cleaning_v1/`. A full run may replace previous files in those locations; it never writes back to Part 1 data.

### B0. Paths and input gates for cleaning

In [ ]:
# Part B uses the same PROJECT_ROOT, manifest and curated Financials as Part A.
# This cell intentionally does not mount Drive or reinstall packages.
QA_DIR = PROJECT_ROOT / 'outputs/part2/raw_qa'
CLEAN_DIR = PROJECT_ROOT / 'data_processed/clean_panel_v1'
EVENT_DIR = PROJECT_ROOT / 'data_processed/part2_event_reference_v1'
REPORT_DIR = PROJECT_ROOT / 'outputs/part2/cleaning_v1'

for _src in (FIN_DIR, MANIFEST_PATH, QA_DIR):
    assert _src.exists(), f'Missing Part B input: {_src}'
assert (QA_DIR / 'audit_summary.json').is_file(), 'Part A report missing: run the 03A export cell first'
for _target in (CLEAN_DIR, EVENT_DIR, REPORT_DIR):
    assert _target != FIN_DIR and _target != RAW_DIR and _target != QA_DIR
    assert _target.relative_to(PROJECT_ROOT).parts[0] in ('data_processed', 'outputs')
    _target.mkdir(parents=True, exist_ok=True)

# Reload this source-of-truth to keep 03B behavior reproducible even after any later edits.
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
entries = manifest.get('entries', {})
prior_qa = json.loads((QA_DIR / 'audit_summary.json').read_text(encoding='utf-8'))
YEAR_FILES = sorted(FIN_DIR.glob('YEAR=*/data.parquet'), key=lambda f: int(f.parent.name.split('=')[1]))
assert len(YEAR_FILES) == 43, f'Expected 43 source years, got {len(YEAR_FILES)}'
assert int(prior_qa['financial_fields_observed']) == 47
assert int(prior_qa['financial_rows']) == 1682616
assert int(prior_qa['duplicate_bank_quarter_rows']) == 0
assert int(prior_qa['quarters_with_record_mismatch']) == 0
expected_q = [p.end_time.strftime('%Y%m%d') for p in pd.period_range('1984Q1','2026Q2',freq='Q')]
assert len(expected_q) == 170
assert all(entries.get('financials:'+q,{}).get('status') in ('VERIFIED','VERIFIED_SCHEMA_GAP') for q in expected_q)
print('Part B input ready: 43 year files; 47 original fields; 1,682,616 rows')
print('Part 1 source (READ ONLY):', FIN_DIR)
print('Part B outputs:', CLEAN_DIR, EVENT_DIR, REPORT_DIR)
print('Polars:', pl.__version__)

## B1. Explicit anomaly policy before transformations

The policy is deliberately **field-level**, not blanket row deletion. These columns are quantities/balances that should ordinarily be nonnegative. Negative reported values are **flagged** and masked **only in an additional `_QA_VALID` view**, while the original field is preserved for auditing. `NETINC`, `EQ`, `ROA`, `ROE`, `NTLNLS`, `ELNATR`, and other signed financial flows remain unchanged. Extreme ratios are **review flags only**, not automatically winsorized. We do not impute missing values or fill missing quarters.

Note that missingness is often *structural* (e.g. `CBLRIND` absent in earlier decades). Imputation, standardization, winsorization and final feature selection belong in **Part 4's training-only pipeline**, not in this panel.

In [ ]:
BALANCE_NONNEGATIVE_FIELDS = [
    'ASSET','DEP','DEPDOM','COREDEP','DEPINS','DEPUNINS','CHBAL',
    'SC','LNLSGR','LNLSNET','P3ASSET','P9ASSET','NAASSET'
]
# Explicit *review* thresholds, not asserted hard accounting bounds.
RATIO_REVIEW_ABS_THRESHOLDS = {'ROA':100.0, 'ROE':1000.0,
                               'NIMY':100.0, 'RBC1AAJ':1000.0}
DENOMINATOR_REVIEW_FIELDS = ['ASSET','DEP','LNLSGR']
ALLOWED_TERMINAL_GAP_QUARTERS = 4   # diagnostic alignment tolerance, not FDIC definition
STUDY_START = pd.Timestamp('1984-01-01')
STUDY_END = pd.Timestamp('2026-06-30')

cleaning_policies = [
 {'issue':'Missing values (including structurally unavailable fields)',
  'treatment':'Preserve original null; no global imputation or zero fill',
  'reason':'Avoid invented history and training/test leakage',
  'original_changed':False},
 {'issue':'Negative assets, deposits, securities, loans and other listed balances',
  'treatment':'Preserve raw; add *_QA_VALID=null if raw<0; mark QA_NEGATIVE_BALANCE_COUNT',
  'reason':'Conservative variable-level quarantine without discarding distressed banks',
  'original_changed':False},
 {'issue':'Negative NETINC, EQ, ROA, ROE, NTLNLS and other signed flows',
  'treatment':'Preserve raw; do not set to zero or drop rows',
  'reason':'Potential distress and legitimate accounting signs',
  'original_changed':False},
 {'issue':'Extreme financial ratios',
  'treatment':'QA_RATIO_REVIEW_COUNT flag only, no clipping or winsorizing',
  'reason':'Source may reflect tiny denominator or genuine distress; model-time preprocessing only',
  'original_changed':False},
 {'issue':'Zero ratio denominators',
  'treatment':'Flag QA_ZERO_DENOM_COUNT; protect any future division using *_QA_VALID > 0',
  'reason':'Avoid divide-by-zero without erasing the observation',
  'original_changed':False},
 {'issue':'Discontinuous bank-quarter reports',
  'treatment':'Preserve actual observations; never forward-fill absent bank-quarter rows',
  'reason':'Do not fabricate observations or survival history',
  'original_changed':False},
 {'issue':'22 unmatched early-1984 failure records',
  'treatment':'Retain in unmatched event reference; no fabricated Financials matching row',
  'reason':'Cannot join failure event without a bank report',
  'original_changed':False},
 {'issue':'Repeated failure CERT and post-failure financial reporting',
  'treatment':'Quarantine event association in retrospective EVENT_LINK_REVIEW, not guessed',
  'reason':'CERT alone does not establish a single continuous institution lifetime',
  'original_changed':False},
 {'issue':'Non-failure bank disappearance',
  'treatment':'OTHER_EXIT_DATE_SUPPORTED only if inactive snapshot plus nearby ENDEFYMD; otherwise EXIT_UNVERIFIED',
  'reason':'Neither absence from panel nor 2026 ACTIVE proves a historical merger',
  'original_changed':False},
 {'issue':'Historical publication lags',
  'treatment':'REPDTE retained as quarter end; no assertion of instant public availability',
  'reason':'Part 4 must adopt/report conservative as-of information lag',
  'original_changed':False},
]
policy_df=pd.DataFrame(cleaning_policies)
display(policy_df)

## B2. Load verified auxiliary events and diagnosis; no new API calls

`API_RECORD_ID` is unique per source; it is **not** a replacement for `CERT`. Failed institutions are linked by `CERT` **and the failure date**, with verification against the actual sequence of `REPDTE` records. `ACTIVE` and `ENDEFYMD` come from a later snapshot and may be used only for **retrospective exit evidence** — never for historical predictors.

In [ ]:
def load_verified_aux(source):
    info=entries.get(f'aux:{source}:summary',{})
    assert info.get('status') in ('VERIFIED','VERIFIED_SCHEMA_GAP'), (source, info)
    snap=info.get('snapshot')
    folder=AUX_DIR/source/f'snapshot={snap}'
    files=sorted(folder.glob('page_*.parquet'))
    assert files, f'No verified aux pages for {source} at {folder}'
    pieces=[pd.read_parquet(p) for p in files]
    frame=pd.concat(pieces,ignore_index=True)
    assert len(frame)==int(info['api_total']), (source,len(frame),info['api_total'])
    assert 'API_RECORD_ID' in frame.columns
    ident=frame['API_RECORD_ID'].astype('string').str.strip()
    assert ident.notna().all() and ident.ne('').all() and ident.is_unique, f'{source} ID audit failed'
    return frame

failures=load_verified_aux('failures')
institutions=load_verified_aux('institutions')
bank_index=pd.read_csv(QA_DIR/'audit_bank_exit_candidates.csv')
assert len(bank_index)==int(prior_qa['unique_financial_CERTs'])

failures['CERT_KEY']=pd.to_numeric(failures['CERT'],errors='coerce').astype('Int64')
failures['FAILDATE_DT']=pd.to_datetime(failures['FAILDATE'].astype('string').str.strip(),
                                     format='mixed',errors='coerce')
failures['WITHIN_STUDY']=failures['FAILDATE_DT'].between(STUDY_START,STUDY_END,inclusive='both')
institutions['CERT_KEY']=pd.to_numeric(institutions['CERT'],errors='coerce').astype('Int64')
institutions['ENDEFYMD_DT']=pd.to_datetime(institutions['ENDEFYMD'].astype('string').str.strip(),
                                         format='mixed',errors='coerce')
institutions['ACTIVE_NUM']=pd.to_numeric(institutions['ACTIVE'],errors='coerce')

failure_dup_cert=failures[failures['WITHIN_STUDY']&failures['CERT_KEY'].notna()]
event_count_in_study=failure_dup_cert.groupby('CERT_KEY').size().to_dict()
event_count_all=failures[failures['CERT_KEY'].notna()].groupby('CERT_KEY').size().to_dict()
assert institutions['CERT_KEY'].dropna().is_unique, 'Institutions snapshot has duplicate CERT; investigate before join'
inst_by_cert=institutions.set_index('CERT_KEY').to_dict(orient='index')
print('Failures:',len(failures),'| in study:',int(failures['WITHIN_STUDY'].sum()),
      '| missing CERT:',int(failures['CERT_KEY'].isna().sum()))
print('Institutions:',len(institutions),'| bad/sentinel institution end dates:',
      int(institutions['ENDEFYMD_DT'].isna().sum()))
print('Repeated in-window failure CERTs:',sum(v>1 for v in event_count_in_study.values()))

## B3. Targeted chronology review: financial reports before/after each failure

The old QA's `CERT`-only match rate does not prove a valid *as-of* link. This step reads only the two key columns from each year's parquet, and retains rows for CERTs having **an in-study failure**. It then calculates the **last quarter-end strictly before FAILDATE**, the earliest report on/after FAILDATE, and the exact number of such reports. Any multiple event, reporting after failure, or no prior observation goes to a separate review file rather than being silently corrected.

In [ ]:
event_certs=set(int(c) for c in failures.loc[
    failures['WITHIN_STUDY'] & failures['CERT_KEY'].notna(),'CERT_KEY'].unique())
cert_list=sorted(event_certs)
report_by_cert=defaultdict(list)

for i,src in enumerate(YEAR_FILES,1):
    # Column-pruned, filtered Polars read: no 47-field Financials copy is held in memory.
    view=(pl.scan_parquet(str(src),hive_partitioning=False)
          .select(['CERT','REPDTE'])
          .filter(pl.col('CERT').is_in(cert_list))
          .collect())
    for cert,q in view.iter_rows():
        if cert is not None and q is not None:
            report_by_cert[int(cert)].append(int(str(q)))
    if i==1 or i%5==0 or i==len(YEAR_FILES):
        print(f'[Chronology] {i}/{len(YEAR_FILES)} year files',flush=True)

for cert in report_by_cert:
    report_by_cert[cert].sort()

# Old QA already established no duplicated bank-quarter keys.
def quarter_index(x):
    d=pd.Timestamp(x)
    return d.year*4 + (d.quarter-1)

event_rows=[]
for row in failures.itertuples(index=False):
    cert = int(row.CERT_KEY) if pd.notna(row.CERT_KEY) else None
    fail_date = row.FAILDATE_DT
    history = report_by_cert.get(cert,[]) if cert is not None else []
    valid_date=pd.notna(fail_date)
    before_count=after_count=0
    last_pre=first_post=None
    gap=None
    if valid_date and history:
        fail_day=int(fail_date.strftime('%Y%m%d'))
        split=bisect.bisect_left(history,fail_day)
        before_count,after_count=split,len(history)-split
        last_pre=history[split-1] if split else None
        first_post=history[split] if split<len(history) else None
        if last_pre is not None:
            gap=quarter_index(fail_date)-quarter_index(pd.Timestamp(str(last_pre)))

    if not valid_date:
        status='INVALID_FAILURE_DATE'
    elif not bool(row.WITHIN_STUDY):
        status='OUTSIDE_STUDY_WINDOW'
    elif cert is None:
        status='NO_CERT_IN_FAILURE_RECORD'
    elif not history:
        status='NO_FINANCIAL_HISTORY_FOR_CERT'
    elif event_count_all.get(cert,0)>1:
        status='MULTIPLE_FAILURE_EVENTS_SAME_CERT_REVIEW'
    elif before_count==0:
        status='NO_PRIOR_FINANCIAL_REPORT_REVIEW'
    elif after_count>0:
        status='POST_FAILURE_FINANCIALS_PRESENT_REVIEW'
    elif gap is None or not (0<=gap<=ALLOWED_TERMINAL_GAP_QUARTERS):
        status='LONG_LAST_REPORT_TO_FAILURE_GAP_REVIEW'
    else:
        status='ALIGNED_FAILURE_EVENT'

    event_rows.append({
        'API_RECORD_ID':str(row.API_RECORD_ID), 'CERT':cert,
        'FAILDATE':row.FAILDATE,'FAILDATE_DT':fail_date,
        'WITHIN_STUDY':bool(row.WITHIN_STUDY),
        'EVENTS_FOR_CERT_ALL_TIME':int(event_count_all.get(cert,0)) if cert is not None else 0,
        'EVENTS_FOR_CERT_IN_STUDY':int(event_count_in_study.get(cert,0)) if cert is not None else 0,
        'BANK_OBSERVED_REPORTS':len(history),
        'REPORTS_STRICTLY_BEFORE_FAILDATE':before_count,
        'REPORTS_ON_OR_AFTER_FAILDATE':after_count,
        'LAST_REPORT_BEFORE_FAILDATE':str(last_pre) if last_pre is not None else None,
        'FIRST_REPORT_ON_OR_AFTER_FAILDATE':str(first_post) if first_post is not None else None,
        'QUARTER_GAP_FROM_LAST_PRIOR_REPORT':gap,
        'EVENT_ALIGNMENT_STATUS':status,
        'ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL':status=='ALIGNED_FAILURE_EVENT'
    })

event_audit=pd.DataFrame(event_rows)
assert len(event_audit)==len(failures)
assert event_audit['API_RECORD_ID'].is_unique
print('Failure event chronology review:')
print(event_audit[event_audit.WITHIN_STUDY]['EVENT_ALIGNMENT_STATUS'].value_counts().to_string())
print('\nUnmatched events:',int((event_audit.EVENT_ALIGNMENT_STATUS=='NO_FINANCIAL_HISTORY_FOR_CERT').sum()))
print('IMPORTANT: ambiguity is quarantined, not auto-corrected.')

## B4. Evidence-ranked exit classification (retrospective table, **not** model features)

Classification is bank-specific, taking into account whether a failure event has a **credible prior Financials sequence**. A missing final report is **not** automatically a merger. If no linked failure exists, an inactive *later institution snapshot* plus `ENDEFYMD` near the last Financials quarter supports **an other-exit date**, **not the underlying cause**. Without the History endpoint, a merger, voluntary closure or charter change cannot be distinguished reliably.

**Status hierarchy:** `FAILURE_CONFIRMED_ALIGNED`, `FAILURE_EVENT_LINK_REVIEW`, `RIGHT_CENSORED_AT_END`, `OTHER_EXIT_DATE_SUPPORTED`, `EXIT_UNVERIFIED`. Do not feed retrospective statuses into features.

In [ ]:
events_in_study=event_audit[event_audit.WITHIN_STUDY & event_audit.CERT.notna()]
by_bank_events={int(cert):part for cert,part in events_in_study.groupby('CERT',sort=False)}

bank_decisions=[]
for b in bank_index.itertuples(index=False):
    cert=int(b.CERT)
    first_q=str(int(b.first_quarter))
    last_q=str(int(b.last_quarter))
    obs=int(b.observed_quarters)
    gap_q=int(b.gap_quarters)
    ev=by_bank_events.get(cert)
    ev_n=0 if ev is None else len(ev)
    snapshot=inst_by_cert.get(cert)
    active=None if snapshot is None else snapshot.get('ACTIVE_NUM')
    end_date=pd.NaT if snapshot is None else snapshot.get('ENDEFYMD_DT',pd.NaT)
    last_end=pd.Timestamp(last_q)
    near_end=(pd.notna(end_date) and last_end<=end_date<=last_end+pd.DateOffset(months=12))
    # Do not confuse an all-NaT end-date sentinel with a known closure.
    if ev_n:
        if (ev_n==1 and
            ev.iloc[0]['EVENT_ALIGNMENT_STATUS']=='ALIGNED_FAILURE_EVENT'):
            exit_status='FAILURE_CONFIRMED_ALIGNED'
            evidence='FDIC_FAILURE_DATE_AND_PRE_FAILURE_FINANCIALS'
            date=ev.iloc[0]['FAILDATE_DT']
            event_id=ev.iloc[0]['API_RECORD_ID']
        else:
            exit_status='FAILURE_EVENT_LINK_REVIEW'
            evidence='FAILURE_CERT_MATCH_BUT_EVENT_CHRONOLOGY_UNCERTAIN'
            date=pd.NaT
            event_id=None
    elif last_q=='20260630':
        exit_status='RIGHT_CENSORED_AT_END'
        evidence='LAST_FINANCIAL_QUARTER_EQUALS_STUDY_END'
        date=pd.NaT
        event_id=None
    elif event_count_all.get(cert,0)>0:
        exit_status='EXIT_UNVERIFIED'
        evidence='FAILURE_RECORD_OUTSIDE_STUDY_WINDOW_CERT_REUSE_REVIEW'
        date=pd.NaT
        event_id=None
    elif pd.notna(active) and int(active)==0 and near_end:
        exit_status='OTHER_EXIT_DATE_SUPPORTED'
        evidence='INSTITUTIONS_SNAPSHOT_INACTIVE_AND_END_DATE_WITHIN_12_MONTHS'
        date=end_date
        event_id=None
    else:
        exit_status='EXIT_UNVERIFIED'
        evidence=('INSTITUTIONS_SNAPSHOT_ACTIVE_BUT_REPORTING_STOPPED' if
                  pd.notna(active) and int(active)==1 else
                  'NO_VERIFIED_EVENT_CAUSE_OR_CLOSE_DATE')
        date=pd.NaT
        event_id=None

    bank_decisions.append({
      'CERT':cert, 'FIRST_FINANCIAL_QUARTER':first_q,
      'LAST_FINANCIAL_QUARTER':last_q,'FINANCIAL_QUARTERS_OBSERVED':obs,
      'INTERIOR_MISSING_QUARTERS':gap_q,'STUDY_FAILURE_EVENT_COUNT':ev_n,
      'RETROSPECTIVE_EXIT_CLASS':exit_status,
      'EVIDENCE_LEVEL':('HIGH' if exit_status=='FAILURE_CONFIRMED_ALIGNED'
                         else 'DATE_SUPPORTED_ONLY' if exit_status=='OTHER_EXIT_DATE_SUPPORTED'
                         else 'UNRESOLVED' if 'REVIEW' in exit_status or exit_status=='EXIT_UNVERIFIED'
                         else 'CENSORED'),
      'EVIDENCE_REASON':evidence,'SUPPORTED_EVENT_DATE':date,
      'LINKED_FAILURE_API_RECORD_ID':event_id,
      'INSTITUTIONS_ACTIVE_SNAPSHOT_ONLY':active,
      'INSTITUTIONS_ENDEFYMD_SNAPSHOT_ONLY':end_date,
      'EVENT_METADATA_IS_RETROSPECTIVE_NOT_FEATURE':True,
    })

bank_outcomes=pd.DataFrame(bank_decisions)
assert len(bank_outcomes)==len(bank_index)
assert bank_outcomes.CERT.is_unique
print(bank_outcomes.RETROSPECTIVE_EXIT_CLASS.value_counts().to_string())
print('\nHistory metadata loaded? No: Part 1 config DOWNLOAD_HISTORY=False.')
print('OTHER_EXIT_DATE_SUPPORTED does NOT assert an actual merger or a verified exit cause.')

## B5. Create the clean bank-quarter panel with Polars, one year at a time

Read each immutable source Parquet into Polars, preserve all original 47 columns, add `SOURCE_ROW_IN_YEAR_FILE` for traceability, create `*_QA_VALID` views for explicitly nonnegative balances, and retain row-level anomaly flags. No historical closure date or future target field is attached. Each year is written to its own new Parquet file in `data_processed/clean_panel_v1/YEAR=YYYY/`.

This operation may take several minutes on Drive. It never calls the FDIC API and **never deletes Part 1 inputs**.

In [ ]:
orig_schema_names=pq.ParquetFile(YEAR_FILES[0]).schema_arrow.names
assert len(orig_schema_names)==47, f'Unexpected source schema ({len(orig_schema_names)} columns)'
existing_masks=[f for f in BALANCE_NONNEGATIVE_FIELDS if f in orig_schema_names]
existing_ratios={k:v for k,v in RATIO_REVIEW_ABS_THRESHOLDS.items() if k in orig_schema_names}
assert {'CERT','REPDTE'} <= set(orig_schema_names)

# Fail fast if someone accidentally introduced target data into financial predictor outputs.
FORBIDDEN_FEATURE_COLS={'FAILDATE','FAILDATE_DT','NEXT_FAILURE_DATE',
                        'RETROSPECTIVE_EXIT_CLASS','ACTIVE','ENDEFYMD','FAILURE_NEXT_4Q'}
assert not (FORBIDDEN_FEATURE_COLS & set(orig_schema_names)), 'Future/status field found in input Financials'

def transform_one_year(raw, src_year):
    assert raw.columns == orig_schema_names, 'Source schema changed; inspect before proceeding'
    df=raw.with_row_index('SOURCE_ROW_IN_YEAR_FILE')
    masked=[]
    for name in existing_masks:
        flag=(pl.col(name)<0).fill_null(False)
        df=df.with_columns(
            pl.when(flag).then(None).otherwise(pl.col(name)).alias(f'{name}_QA_VALID')
        )
        masked.append(flag.cast(pl.Int8))

    ratioflags=[(pl.col(name).abs()>limit).fill_null(False).cast(pl.Int8)
                for name,limit in existing_ratios.items()]
    zero_flags=[(pl.col(x)==0).fill_null(False).cast(pl.Int8)
                for x in DENOMINATOR_REVIEW_FIELDS if x in raw.columns]
    any_false=pl.lit(0,dtype=pl.Int32)
    df=df.with_columns([
        (pl.sum_horizontal(masked) if masked else any_false).cast(pl.Int32).alias('QA_NEGATIVE_BALANCE_COUNT'),
        (pl.sum_horizontal(ratioflags) if ratioflags else any_false).cast(pl.Int32).alias('QA_RATIO_REVIEW_COUNT'),
        (pl.sum_horizontal(zero_flags) if zero_flags else any_false).cast(pl.Int32).alias('QA_ZERO_DENOM_COUNT'),
        pl.col('REPDTE').str.strptime(pl.Date,format='%Y%m%d',strict=True).alias('REPDTE_DATE'),
        pl.lit('FDIC_FINANCIALS_PART1_YEAR_PARQUET').alias('SOURCE_DATASET'),
        pl.lit(int(src_year)).alias('SOURCE_YEAR'),
    ])
    # Original fields are kept bit-for-bit as Polars values, not overwritten.
    assert df.select(orig_schema_names).equals(raw), f'{src_year}: raw column mutation detected'
    return df

per_year_results=[]
mask_counts=Counter()
ratio_review_total=0
zero_denom_total=0
anomaly_samples=[]
SAMPLE_PER_FIELD=30
sample_by_field=Counter()

for i,f in enumerate(YEAR_FILES,1):
    year=int(f.parent.name.split('=')[1])
    raw=pl.read_parquet(str(f),hive_partitioning=False)
    cleaned=transform_one_year(raw,year)

    # Strong invariants: no artificial or deleted rows; no invented bank-quarter keys.
    assert cleaned.height==raw.height
    assert cleaned.n_unique(subset=['CERT','REPDTE'])==raw.height, (
        f'Nonunique bank-quarter keys in {year}')
    assert (cleaned.select((pl.col('REPDTE').str.slice(0,4)!=str(year)).sum()).item()==0)
    assert all(col in cleaned.columns for col in orig_schema_names)
    assert not (FORBIDDEN_FEATURE_COLS & set(cleaned.columns))

    counts={}
    for c in existing_masks:
        val=int(cleaned.select((pl.col(c)<0).fill_null(False).sum()).item())
        counts[c]=val
        mask_counts[c]+=val
        if val and sample_by_field[c] < SAMPLE_PER_FIELD:
            remaining=SAMPLE_PER_FIELD-sample_by_field[c]
            s=(cleaned.filter((pl.col(c)<0).fill_null(False))
                      .select(['CERT','REPDTE',c,f'{c}_QA_VALID'])
                      .head(remaining).to_dicts())
            for rr in s:
                anomaly_samples.append({'field':c,'CERT':rr['CERT'],'REPDTE':rr['REPDTE'],
                    'raw_value':rr[c],'QA_VALID_value':rr[f'{c}_QA_VALID'],
                    'handling':'RAW_RETAINED_VALID_VIEW_NULL'})
            sample_by_field[c]+=len(s)

    year_ratio=int(cleaned['QA_RATIO_REVIEW_COUNT'].sum())
    year_zero=int(cleaned['QA_ZERO_DENOM_COUNT'].sum())
    ratio_review_total+=year_ratio
    zero_denom_total+=year_zero

    output_file=CLEAN_DIR/f'YEAR={year}'/'data.parquet'
    output_file.parent.mkdir(parents=True,exist_ok=True)
    # Write locally first; transfer to Drive only after a complete Parquet file exists.
    with tempfile.TemporaryDirectory(prefix='fdic_clean_') as td:
        stage=Path(td)/'data.parquet'
        cleaned.write_parquet(str(stage),compression='zstd',statistics=True)
        partial=output_file.with_suffix('.parquet.partial')
        shutil.copy2(stage,partial)
        partial.replace(output_file)
    check=pq.ParquetFile(output_file)
    assert int(check.metadata.num_rows)==raw.height, f'Output truncated: {year}'

    per_year_results.append({
        'year':year,'raw_rows':raw.height,'clean_rows':cleaned.height,
        'raw_columns':len(orig_schema_names),'clean_columns':cleaned.width,
        'negative_balance_values_masked_in_QA_VALID_views':sum(counts.values()),
        'extreme_ratio_review_flags':year_ratio,
        'zero_denominator_review_flags':year_zero,
        'raw_values_preserved':True,
        'source_path':str(f.relative_to(PROJECT_ROOT)),
        'clean_path':str(output_file.relative_to(PROJECT_ROOT)),
        'source_parquet_bytes':f.stat().st_size,
        'output_parquet_bytes':output_file.stat().st_size,
    })
    del raw,cleaned
    if i==1 or i%5==0 or i==len(YEAR_FILES):
        print(f'[Clean panel] {i}/{len(YEAR_FILES)} year files | '
              f'{sum(x["clean_rows"] for x in per_year_results):,} rows',flush=True)

per_year_df=pd.DataFrame(per_year_results)
assert int(per_year_df['clean_rows'].sum())==int(prior_qa['financial_rows'])
assert int(per_year_df['clean_rows'].sum())==int(per_year_df['raw_rows'].sum())
print('CLEAN PANEL COMPLETE | rows:',f'{per_year_df.clean_rows.sum():,}',
      '| year partitions:',len(per_year_df),'| original fields preserved:',len(orig_schema_names))

## B6. Save independent event/outcome references and produce before/after audit

The feature-safe panel stores only point-in-time financials, source row provenance and data-quality indicators. The separate event reference table includes future failure dates and current snapshot evidence, which must not become predictors. Review cases remain explicitly unresolved.

In [ ]:
def atomic_parquet(frame, path):
    path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='fdic_event_') as td:
        stage=Path(td)/'data.parquet'
        # pandas PyArrow writer: retain nullable integer/date types.
        frame.to_parquet(stage,engine='pyarrow',index=False,compression='zstd')
        partial=path.with_suffix('.parquet.partial')
        shutil.copy2(stage,partial)
        partial.replace(path)

atomic_parquet(event_audit,EVENT_DIR/'failure_event_alignment.parquet')
atomic_parquet(bank_outcomes,EVENT_DIR/'bank_exit_outcomes_retrospective.parquet')

# Explicitly preserve all nonaligned events and bank outcomes as review files.
review_events=event_audit[(event_audit['WITHIN_STUDY']) &
                         (~event_audit['ALIGNMENT_CONFIRMED_FOR_MODEL_LABEL'])].copy()
review_banks=bank_outcomes[bank_outcomes.RETROSPECTIVE_EXIT_CLASS.isin(
                     ['FAILURE_EVENT_LINK_REVIEW','EXIT_UNVERIFIED'])].copy()

per_year_df.to_csv(REPORT_DIR/'qa_before_after_by_year.csv',index=False,encoding='utf-8-sig')
event_audit.to_csv(REPORT_DIR/'failure_event_alignment.csv',index=False,encoding='utf-8-sig')
review_events.to_csv(REPORT_DIR/'failure_event_review_queue.csv',index=False,encoding='utf-8-sig')
bank_outcomes.to_csv(REPORT_DIR/'bank_exit_outcomes_retrospective.csv',index=False,encoding='utf-8-sig')
review_banks.to_csv(REPORT_DIR/'unresolved_bank_exit_review.csv',index=False,encoding='utf-8-sig')
policy_df.to_csv(REPORT_DIR/'cleaning_policy_decisions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(anomaly_samples,columns=['field','CERT','REPDTE','raw_value','QA_VALID_value','handling']).to_csv(
    REPORT_DIR/'negative_balance_case_samples.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'field':k,'negative_values_flagged':v,'treatment':'RAW_KEPT__QA_VALID_NULL'}
              for k,v in sorted(mask_counts.items())]).to_csv(
              REPORT_DIR/'balance_mask_counts.csv',index=False,encoding='utf-8-sig')

# Safe report about whether the *input* QA identified missing reporting quarters.
gaps=pd.read_csv(QA_DIR/'audit_bank_reporting_gaps.csv')
print('Banks with earlier internal gaps preserved (no new rows created):',len(gaps))

# Reconciliation across emitted data and classifications.
clean_files=sorted(CLEAN_DIR.glob('YEAR=*/data.parquet'))
assert len(clean_files)==len(YEAR_FILES), 'Clean output is missing a year or has stale extra year folders'
assert sum(pq.ParquetFile(p).metadata.num_rows for p in clean_files)==int(prior_qa['financial_rows'])
assert bank_outcomes.RETROSPECTIVE_EXIT_CLASS.isna().sum()==0
assert bank_outcomes.CERT.is_unique
assert not ((bank_outcomes.RETROSPECTIVE_EXIT_CLASS=='OTHER_EXIT_DATE_SUPPORTED') &
            (bank_outcomes.STUDY_FAILURE_EVENT_COUNT>0)).any()

checks=[
 ('ALL_SOURCE_ROWS_RETAINED',int(per_year_df.clean_rows.sum())==int(prior_qa['financial_rows'])),
 ('ALL_43_YEAR_PARTITIONS_EMITTED',len(clean_files)==43),
 ('ALL_ORIGINAL_47_FINANCIAL_FIELDS_PRESERVED',bool(per_year_df.raw_values_preserved.all() and len(orig_schema_names)==47)),
 ('NO_DUPLICATE_BANK_QUARTER_KEYS',int(prior_qa['duplicate_bank_quarter_rows'])==0),
 ('BANK_OUTCOMES_ONE_ROW_PER_CERT',bank_outcomes.CERT.is_unique),
 ('FAILURE_API_RECORD_IDS_UNIQUE',event_audit.API_RECORD_ID.is_unique),
 ('UNKNOWN_EXITS_NOT_MISLABELLED_AS_MERGERS',True),
 ('FAILURE_REVIEW_NOT_PROMOTED_TO_CONFIRMED', not bool(set(
    bank_outcomes.loc[bank_outcomes.RETROSPECTIVE_EXIT_CLASS=='FAILURE_EVENT_LINK_REVIEW','CERT']
    ) & set(
    bank_outcomes.loc[bank_outcomes.RETROSPECTIVE_EXIT_CLASS=='FAILURE_CONFIRMED_ALIGNED','CERT']))),
 ('NO_FUTURE_EVENT_COLUMNS_IN_FINANCIAL_PANEL',not (FORBIDDEN_FEATURE_COLS & set(orig_schema_names))),
]
check_df=pd.DataFrame(checks,columns=['gate','pass'])
check_df.to_csv(REPORT_DIR/'part2_clean_panel_acceptance.csv',index=False)
assert check_df['pass'].all(), f'Acceptance failed:\n{check_df[~check_df["pass"]]}'

summary={
 'pipeline':'PART2_CLEAN_PANEL_AND_RETROSPECTIVE_EXIT_REFERENCE_V1',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_financials':str(FIN_DIR.relative_to(PROJECT_ROOT)),
 'clean_panel':str(CLEAN_DIR.relative_to(PROJECT_ROOT)),
 'retrospective_events':str(EVENT_DIR.relative_to(PROJECT_ROOT)),
 'diagnostic_input':str(QA_DIR.relative_to(PROJECT_ROOT)),
 'raw_input_fields_preserved':int(len(orig_schema_names)),
 'rows_before':int(prior_qa['financial_rows']),
 'rows_after':int(per_year_df.clean_rows.sum()),
 'year_partitions':len(clean_files),
 'balance_fields_protected':existing_masks,
 'mask_counts_by_field':dict(mask_counts),
 'ratio_review_flag_instances':ratio_review_total,
 'zero_denominator_flag_instances':zero_denom_total,
 'failure_events_by_alignment':event_audit[event_audit.WITHIN_STUDY].EVENT_ALIGNMENT_STATUS.value_counts().to_dict(),
 'banks_by_retrospective_outcome':bank_outcomes.RETROSPECTIVE_EXIT_CLASS.value_counts().to_dict(),
 'failure_review_events':len(review_events),
 'bank_exit_review_cases':len(review_banks),
 'banks_with_original_reporting_gaps':len(gaps),
 'outcome_policy':'Retrospective outcomes separate from predictor panel; ambiguous CERT/timing links unresolved',
 'other_exit_policy':'Institutions snapshot end date provides date support only, not a verified cause (not necessarily merger)',
 'history_endpoint_note':'History not acquired in Part 1; finer merger/charter classifications are not warranted',
 'publication_timing_note':'REPDTE is a quarter-end, not a public release time; Part 4 must impose conservative feature lag',
 'no_source_data_modified':True,
 'software':{'python':sys.version.split()[0], 'polars':pl.__version__,
             'pandas':pd.__version__, 'platform':platform.platform()},
 'acceptance_passed':bool(check_df['pass'].all()),
}
(REPORT_DIR/'cleaning_summary.json').write_text(
    json.dumps(summary,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
with zipfile.ZipFile(REPORT_DIR/'FDIC_Part2_Clean_Panel_Reports.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(REPORT_DIR.iterdir()):
        if p.is_file() and p.name!='FDIC_Part2_Clean_Panel_Reports.zip':
            zf.write(p,arcname=p.name)
print('\nPART 2 CLEANING REPORTS:',REPORT_DIR)
print('UPLOAD FOR REVIEW:',REPORT_DIR/'FDIC_Part2_Clean_Panel_Reports.zip')
print('\nACCEPTANCE GATES:')
print(check_df.to_string(index=False))
print('\nEVENT STATUS:')
print(event_audit[event_audit.WITHIN_STUDY].EVENT_ALIGNMENT_STATUS.value_counts().to_string())
print('\nEXIT STATUS:')
print(bank_outcomes.RETROSPECTIVE_EXIT_CLASS.value_counts().to_string())

## B7. Interpretation for final report and Part 3/4 handoff

1. **Raw-to-clean audit:** compare `qa_before_after_by_year.csv`, `balance_mask_counts.csv`, and `cleaning_policy_decisions.csv`. All 1,682,616 rows and 47 original fields must be preserved. The `*_QA_VALID` columns encode *field-level* unusability of negative balances, rather than erasing original evidence.
2. **Failure classification:** only `ALIGNED_FAILURE_EVENT` has sufficiently unambiguous CERT/date chronology under the **explicit conservative rule**. All other in-window failure events remain in `failure_event_review_queue.csv`; do not call them safe negatives.
3. **Other exits:** `OTHER_EXIT_DATE_SUPPORTED` is historical **closure-date corroboration**, not proven merger. More granular exit cause requires downloaded and validated History/event evidence. `EXIT_UNVERIFIED` is **unknown**, never an automatic non-failure for partially observed horizons.
4. **Time leakage:** the clean Financials panel intentionally has **no** failure dates, contemporary institution status, or future outcomes. The separate retrospective event reference is suitable for carefully constructing Part 4 labels. Restrict inputs at time *q* to information available by prediction time, with a justified reporting lag.
5. **Part 3:** reuse `transform_one_year(raw, year)` for Polars-vs-pandas parity/benchmark; compare exact same inputs, rules and local disk, then reproduce the same field-level flags in PySpark.

**Deliverable for review:** `outputs/part2/cleaning_v1/FDIC_Part2_Clean_Panel_Reports.zip`. The detailed result counts can only be known after execution against the user's Drive Parquet data. Do **not** re-run Notebook 02 or overwrite raw files.

---
# Part C — History-based exit verification (03C: verified separate extension)

**Method:** FDIC History was analyzed in the independently executed **Notebook 03C**, rather than embedded into this notebook, to avoid re-downloading or reprocessing 43 years of History whenever Parts A/B are rerun. History is retrospective **outcome evidence** only; it is **not** a historical predictor and must not be merged into the financial feature columns.

**Results from the completed 03C run (1984–2026Q2):** 506,824 History records across 43 verified yearly slices; **14,346 banks with supported unassisted-merger evidence** under the stated identity/code/chronology rules. The revised classification over 24,064 banks was:

| Retrospective outcome class | Banks |
|:--|--:|
| `MERGER_VERIFIED_HISTORY` | 14,346 |
| `FAILURE_CONFIRMED_ALIGNED` | 2,768 |
| `RIGHT_CENSORED_AT_END` | 4,307 |
| `OTHER_EXIT_DATE_SUPPORTED` | 2,038 |
| `FAILURE_EVENT_LINK_REVIEW` | 448 |
| `EXIT_UNVERIFIED` | 136 |
| `OTHER_EXIT_HISTORY_REVIEW` | 21 |
| **Total** | **24,064** |

**Data lineage / links within the project** (under `MyDrive/FDIC_Bank_Failure_Project/`):

- **Updated bank-level classification (CSV):** `outputs/part2/history_exit_v1/bank_exit_outcomes_with_history.csv`
- **Updated classification (Parquet):** `data_processed/part2_history_exit_reference_v1/bank_exit_outcomes_with_history.parquet`
- **Transition table:** `outputs/part2/history_exit_v1/exit_class_transition_matrix.csv`
- **Review queue:** `outputs/part2/history_exit_v1/history_event_review_queue.csv`
- **Summary and 7 acceptance checks:** `outputs/part2/history_exit_v1/history_exit_summary.json`, `history_acceptance_gates.csv`
- **Standalone pipeline:** `03C_FDIC_History_Exit_Verification.ipynb`

**Scope and caution:** These numbers reflect the recorded 03C outputs, not independent adjudication of every bank. `MERGER_VERIFIED_HISTORY` is an evidence-based classification subject to the research code/temporal rules. `OTHER_EXIT_DATE_SUPPORTED` is **not** a verified merger; assisted merger, ambiguous failure links and other unresolved exits remain flagged. The feature-safe clean bank–quarter panel remains as produced by Part B; **for Part 4 label construction, prefer 03C's updated outcome reference over 03B's original baseline**, and implement time-window censoring / competing-exit rules at the bank–quarter level.